# HVAC Takeoff — v11 Training on Kaggle

**v11 changes vs v10:**
- Adds 315 LS-verified tiles from 6 reviewed projects (Sola, Erewhon, Bungalow, Anaheim 82, BMO, Saint Mary's)
- Same 33 classes, same imgsz/batch/aug as v10 — only the data changed
- Should fix the AD-GRD ↔ AD-T-BAR SUPPLY collapse (89/89 May-5 relabels were this confusion)

**Setup:**
1. Upload `yolo_dataset_v11.zip` as a Kaggle Dataset → name `hvac-yolo-dataset-v11`
2. Settings → Accelerator → **GPU P100**
3. Right panel → Add Data → search `hvac-yolo-dataset-v11`
4. Run all cells (initialize from v10 weights for faster convergence)

In [ ]:
# Step 1: Install
!pip install ultralytics -q

In [ ]:
# Step 2: Copy dataset from Kaggle input to writable directory
import yaml, os, shutil, glob

candidates = glob.glob('/kaggle/input/**/yolo_dataset_v11', recursive=True)
if not candidates:
    print('ERROR: yolo_dataset_v11 folder not found in /kaggle/input/')
    for root, dirs, files in os.walk('/kaggle/input/'):
        print(f'  {root}/')
        if len(root.replace('/kaggle/input/', '').split('/')) > 4:
            continue
    raise FileNotFoundError('Dataset not found')

src = candidates[0]
dst = '/kaggle/working/yolo_dataset_v11'
print(f'Found dataset at: {src}')

if os.path.exists(dst):
    shutil.rmtree(dst)
print('Copying to writable directory (this takes ~2-3 min)...')
shutil.copytree(src, dst)

config_path = f'{dst}/dataset.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)
config['path'] = dst
with open(config_path, 'w') as f:
    yaml.dump(config, f)

for cache in ['train.cache', 'val.cache']:
    p = f'{dst}/labels/{cache}'
    if os.path.exists(p):
        os.remove(p)

train = len(os.listdir(f'{dst}/images/train'))
val = len(os.listdir(f'{dst}/images/val'))
print()
print(f'Dataset ready: {train} train, {val} val')
print(f'Classes: {len(config["names"])}')
for i, n in config['names'].items():
    print(f'  {i}: {n}')

In [ ]:
# Step 3: Stage v10 weights as the starting checkpoint (faster convergence than yolov8s.pt cold start).
# Upload hvac_yolov8s_v10.pt as a Kaggle Dataset (e.g., 'hvac-v10-weights') and Add Data, OR
# fall back to yolov8s.pt if v10 weights aren't attached.
import os, glob

v10_candidates = glob.glob('/kaggle/input/**/hvac_yolov8s_v10.pt', recursive=True)
if v10_candidates:
    INIT_WEIGHTS = v10_candidates[0]
    print(f'Initializing from v10 weights: {INIT_WEIGHTS}')
else:
    INIT_WEIGHTS = 'yolov8s.pt'
    print('v10 weights not found in /kaggle/input/ — using yolov8s.pt cold start')

In [ ]:
# Step 4: Train v11 with checkpoint callback
from ultralytics import YOLO
import shutil, os

model = YOLO(INIT_WEIGHTS)

def save_checkpoint(trainer):
    best = '/kaggle/working/runs/hvac_v11/weights/best.pt'
    if os.path.exists(best):
        shutil.copy(best, '/kaggle/working/hvac_yolov8s_v11.pt')
        print(f'  [checkpoint saved: epoch {trainer.epoch + 1}]')

model.add_callback('on_fit_epoch_end', save_checkpoint)

results = model.train(
    data='/kaggle/working/yolo_dataset_v11/dataset.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    device=0,
    workers=2,
    project='/kaggle/working/runs',
    name='hvac_v11',
    exist_ok=True,
    flipud=0.0,
    mosaic=0.5,
    scale=0.3,
    save_period=1,
)

print()
print('Training complete!')
print(f'Final model at: /kaggle/working/hvac_yolov8s_v11.pt')

In [ ]:
# Step 5: Check results
import pandas as pd

df = pd.read_csv('/kaggle/working/runs/hvac_v11/results.csv')
df.columns = [c.strip() for c in df.columns]
best = df.loc[df['metrics/mAP50(B)'].idxmax()]
print(f"Best epoch: {int(best['epoch'])}")
print(f"Precision: {best['metrics/precision(B)']:.1%}")
print(f"Recall:    {best['metrics/recall(B)']:.1%}")
print(f"mAP50:     {best['metrics/mAP50(B)']:.1%}")
print(f"mAP50-95:  {best['metrics/mAP50-95(B)']:.1%}")

print()
print('Last 10 epochs:')
print(df[['epoch','metrics/precision(B)','metrics/recall(B)','metrics/mAP50(B)']].tail(10).to_string(index=False))

In [ ]:
# Step 6: Save model
import shutil

best_pt = '/kaggle/working/runs/hvac_v11/weights/best.pt'
out_path = '/kaggle/working/hvac_yolov8s_v11.pt'
shutil.copy(best_pt, out_path)
print(f'Model saved: {out_path}')
print(f'Size: {os.path.getsize(out_path)/1024/1024:.1f} MB')
print()
print('Download from the Output tab on the right panel →')
print('Then put it in hvac-takeoff-tool/models/')